# Funding Carry 02, Stage A — can funding be predicted, by how much, and does the volatility engine help?

Pre-registration: `discussions/Funding_Carry/003_claude_fc02_stage_a_preregistration.md`. Code: `src/data_engineering/modeling/funding_predict.py`.

- **Target:** next-7-day annualized funding $y_7$ (primary); $y_1$, $y_{30}$ secondary. One row per asset and day at 00:05 UTC.
- **Blocks, added in order (asset fixed effects in every model):** persistence P → premium PR → momentum M → open interest and positioning OI → breadth B → naive volatility V → **engine E** (AM-T-HAR-Blend level, VD18 HAR_4 refitted on Binance spot, TRAIN bars only).
- **Split:** TRAIN origins before 2025-10-01 (target window inside TRAIN), TEST from 2025-10-01. Out-of-sample $R^2$ is relative to asset fixed effects; nested gains use a 14-day date-block bootstrap.
- **Engine checks:** (a) next-24h / next-7d spot RV: QLIKE of ENGINE vs NAIVE vs HAR_D; (b) does E add to the funding forecast beyond V?

Environment: `FC02_SYNTHETIC=1` (smoke run), `FC02_ASSETS` (comma list; default the FC 01 universe).

In [ ]:
from pathlib import Path
import json, os, sys, tempfile, time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from dotenv import load_dotenv

warnings.filterwarnings('ignore', category=RuntimeWarning)
pd.set_option('display.max_columns', 40); pd.set_option('display.width', 250)
load_dotenv()
PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / 'src' / 'data_engineering').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / 'src'))
from data_engineering.modeling import funding_predict as fp, funding_carry as fc, markov_har_variance as v18

SYNTHETIC = os.getenv('FC02_SYNTHETIC', '').strip() == '1'
ASSETS = [a.strip().upper() for a in os.getenv('FC02_ASSETS', ','.join(fc.UNIVERSE)).split(',') if a.strip()]
REGISTERED_RUN = (not SYNTHETIC) and ASSETS == list(fc.UNIVERSE)
if SYNTHETIC:
    OUT = Path(tempfile.mkdtemp(prefix='fc02_'))
else:
    DATA_ROOT = Path(os.environ['DERIBIT_DATA_ROOT']).expanduser().resolve()
    BIN = Path(os.getenv('BINANCE_UM_DIR') or DATA_ROOT.parent / 'Binance').expanduser().resolve()
    OUT = PROJECT_ROOT / 'notebooks' / 'Funding_Carry' / '02'
OUT.mkdir(parents=True, exist_ok=True)
print(fp.VERSION, '| registered run' if REGISTERED_RUN else '| NOT the registered run', '| out', OUT)
print(json.dumps(fp.registration(), indent=1, default=str))

## 1. Panel: features and targets per asset-day; engine forecasts

In [ ]:
T0 = time.time(); parts, raws, cov = [], [], []
if SYNTHETIC:
    S, P, F, M = fp.synthetic(560, seed=7, assets=('AAA', 'BBB', 'CCC', 'DDD', 'EEE'))
    src = {a: (S[a], P[a], F[a], M[a]) for a in S}
else:
    src = {}
    for a in ASSETS:
        sym = f'{a}USDT'
        pp, sp, fu, me = (BIN / f'{sym}_um_1m.parquet', BIN / 'basis' / 'spot' / f'{sym}_spot_1m.parquet',
                          BIN / 'basis' / 'funding' / f'{sym}_funding.parquet', BIN / 'basis' / 'metrics' / f'{sym}_metrics.parquet')
        if not (pp.exists() and sp.exists() and fu.exists()):
            cov.append({'asset': a, 'status': 'missing spot/perp/funding'}); continue
        src[a] = (fc.load_close(sp), fc.load_close(pp), fc.load_funding(fu), fp.load_metrics(me) if me.exists() else None)
for a, (s1, p1, f1, m1) in src.items():
    d = fp.asset_daily(s1, p1, f1, m1); d['asset'] = a; parts.append(d)
    raws.append(fp.spot_returns_5m(s1, a))
    cov.append({'asset': a, 'status': 'ok', 'first': d['b'].min(), 'last': d['b'].max(), 'rows': len(d),
                'metrics': m1 is not None, 'oi_rows': int(d['oi_chg7'].notna().sum())})
PANEL = fp.add_breadth(pd.concat(parts, ignore_index=True))
del parts
COV = pd.DataFrame(cov); COV.to_csv(OUT / '02_coverage.csv', index=False); display(COV)
print(f'panel {PANEL.shape}  {time.time()-T0:.0f}s')
RAW = pd.concat(raws, ignore_index=True); del raws
ENG, ENG_FITS = fp.engine_forecasts(RAW, PANEL.rename(columns={'asset': 'underlying'}), v18)
del RAW
PANEL['leng1'], PANEL['leng7'] = ENG['leng1'].to_numpy(), ENG['leng7'].to_numpy()
print(f'engine done {time.time()-T0:.0f}s | leng1 coverage {PANEL.leng1.notna().mean():.3f}')
(OUT / '02_engine_fits.json').write_text(json.dumps(ENG_FITS, indent=1, default=str))
PANEL.drop(columns=['S']).to_csv(OUT / '02_panel.csv.gz', index=False, float_format='%.6g')

## 2. Which blocks are usable (OI needs ≥ 80% coverage of otherwise-complete rows)

In [ ]:
others = [c for b in fp.ORDER if b != 'OI' for c in fp.BLOCKS[b]]
base_rows = PANEL.dropna(subset=others + [fp.PRIMARY])
oi_share = float(base_rows[fp.BLOCKS['OI']].notna().all(axis=1).mean()) if len(base_rows) else 0.0
BLOCKS_USED = [b for b in fp.ORDER if b != 'OI' or oi_share >= fp.OI_MIN_SHARE]
print('OI coverage', round(oi_share, 3), '-> blocks used', BLOCKS_USED)

## 3. Nested sequence, unique contributions and the engine-for-funding test, per target

In [ ]:
SEQ, UNI, EF, TESTS = [], [], [], {}
for y in fp.TARGETS:
    r = fp.run_models(PANEL, y, BLOCKS_USED)
    SEQ.append(r['table']); TESTS[y] = r
    UNI.append(fp.unique_contribution(PANEL, y, BLOCKS_USED))
    EF.append(fp.engine_vs_naive_for_funding(PANEL, y, [b for b in BLOCKS_USED if b not in ('V', 'E')]))
SEQ, UNI, EF = pd.concat(SEQ), pd.concat(UNI), pd.concat(EF)
SEQ.to_csv(OUT / '02_sequence.csv', index=False); UNI.to_csv(OUT / '02_unique.csv', index=False); EF.to_csv(OUT / '02_engine_for_funding.csv', index=False)
display(SEQ.round(4)); display(UNI.round(4)); display(EF.round(4))

## 4. Volatility-engine check: next-24h and next-7d realized variance (QLIKE)

In [ ]:
VC = fp.vol_check(PANEL); VC.to_csv(OUT / '02_vol_check.csv', index=False); display(VC.round(4))

## 5. How much: quintiles of predicted next-7d funding (TEST), per-asset fit, coefficients

In [ ]:
te = TESTS[fp.PRIMARY]['test']; seq7 = SEQ[SEQ.target == fp.PRIMARY].reset_index(drop=True)
best = seq7.loc[seq7['r2_oos_vs_FE'].idxmax(), 'model']
Q = pd.concat([fp.quintiles(te, 'pred_P').assign(model='P'), fp.quintiles(te, f'pred_{best}').assign(model=best)])
Q.to_csv(OUT / '02_quintiles.csv', index=False); display(Q.round(4))
pa = te.groupby('asset').apply(lambda g: pd.Series({'n': len(g), 'r2_P': fp.r2_oos(g.y_w.to_numpy(), g.pred_P.to_numpy(), g.pred_FE.to_numpy()),
     'r2_best': fp.r2_oos(g.y_w.to_numpy(), g[f'pred_{best}'].to_numpy(), g.pred_FE.to_numpy()),
     'mean_y7': g.y_w.mean()}))
pa.to_csv(OUT / '02_per_asset.csv'); display(pa.round(3).sort_values('r2_best'))
coef = TESTS[fp.PRIMARY]['fits'][best]; C = pd.DataFrame({'term': coef['cols'], 'beta': coef['beta']})
C = C[~C.term.str.startswith('fe_')]; C.to_csv(OUT / '02_coefficients_best.csv', index=False); display(C.round(5))

## 6. Verdict

In [ ]:
DEC = fp.decide(seq7, EF[EF.target == fp.PRIMARY].reset_index(drop=True), VC)
DEC.update({'version': fp.VERSION, 'registered_run': REGISTERED_RUN, 'synthetic': SYNTHETIC, 'blocks_used': BLOCKS_USED,
            'oi_coverage': oi_share, 'n_test_y7': int(seq7.n_test.iloc[0]), 'n_train_y7': int(seq7.n_train.iloc[0]),
            'r2_oos_by_target': {y: float(SEQ[(SEQ.target == y)].r2_oos_vs_FE.max()) for y in fp.TARGETS}})
(OUT / '02_decision.json').write_text(json.dumps(DEC, indent=1, default=str)); print(json.dumps(DEC, indent=1, default=str))

## 7. Plots: predicted vs realized next-7d funding (TEST) and one asset over time

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14, 5))
ax[0].scatter(100 * te[f'pred_{best}'], 100 * te['y_w'], s=3, alpha=.3); lim = [100 * te.y_w.quantile(.005), 100 * te.y_w.quantile(.995)]
ax[0].plot(lim, lim, 'k--', lw=1); ax[0].set_xlabel('predicted y7 (%/yr)'); ax[0].set_ylabel('realized y7 (%/yr)'); ax[0].set_title(f'TEST, {best}')
a0 = 'BTC' if 'BTC' in te.asset.unique() else te.asset.iloc[0]; g = te[te.asset == a0]
ax[1].plot(g.b, 100 * g.y_w, label='realized next-7d'); ax[1].plot(g.b, 100 * g[f'pred_{best}'], label=best); ax[1].plot(g.b, 100 * g.pred_P, label='P', alpha=.7)
ax[1].axhline(0, c='k', lw=.5); ax[1].legend(); ax[1].set_title(f'{a0}: next-7d funding, %/yr')
plt.tight_layout(); plt.savefig(OUT / '02_pred_vs_real.png', dpi=110); plt.show()